# Waveform Plotting Notebook

This notebook reads acquired waveform data from a CSV file (including raw and filtered channels) and plots them inline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Enable inline plotting
%matplotlib inline

## 1. Load the CSV Data

Set the path to your waveform data CSV file below and load it.

In [ ]:
csv_path = "data/line.csv"  # Change this to your file path
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} points from {csv_path}")
df.head()

## 2. Identify and Parse Columns

Extracts the time axis, determines readable time units, and maps the Raw and Filtered columns to their respective channels.

In [ ]:
# Find time column and scale it
time_col = [col for col in df.columns if "Time" in col][0]
time_axis = df[time_col].to_numpy()
time_unit = time_col.split("(")[-1].split(")")[0].strip() if "(" in time_col else "s"

# Determine readable time scale units
max_time = np.max(np.abs(time_axis))
time_multiplier = 1.0
plot_time_unit = time_unit

if max_time < 1e-6:
    time_multiplier = 1e9
    plot_time_unit = "ns"
elif max_time < 1e-3:
    time_multiplier = 1e6
    plot_time_unit = "µs"
elif max_time < 1.0:
    time_multiplier = 1e3
    plot_time_unit = "ms"

scaled_time = time_axis * time_multiplier
print(f"Time scale: {plot_time_unit} (multiplied by {time_multiplier})")

# Map columns to channels
raw_col_map = {}
filtered_col_map = {}
yunit = "V"

for col in df.columns:
    if "_Raw" in col:
        ch = col.split("_Raw")[0]
        raw_col_map[ch] = col
        # Extract voltage/amplitude unit
        if "(" in col:
            yunit = col.split("(")[-1].split(")")[0].strip()
    elif "_Filtered" in col:
        ch = col.split("_Filtered")[0]
        filtered_col_map[ch] = col
    elif col != time_col and not raw_col_map:
        # Fallback if no raw suffixes exist
        raw_col_map[col] = col

print(f"Raw channels found: {list(raw_col_map.keys())}")
print(f"Filtered channels found: {list(filtered_col_map.keys())}")

## 3. Plot All Channels (Raw Data)

Plots all raw signal channels on a single figure.

In [ ]:
plt.figure(figsize=(12, 6))
for ch, col in raw_col_map.items():
    plt.plot(scaled_time, df[col], label=ch)

plt.title("All Channels - Raw Signals")
plt.xlabel(f"Time ({plot_time_unit})")
plt.ylabel(f"Amplitude ({yunit})")
plt.grid(True)
plt.legend()
plt.show()

## 4. Plot Channels 3 & 4

Plots only Channel 3 and Channel 4 (raw signals) on a single figure.

In [ ]:
ch3_ch4_channels = [ch for ch in raw_col_map.keys() if "CH3" in ch or "CH4" in ch]

if ch3_ch4_channels:
    plt.figure(figsize=(12, 6))
    for ch in ch3_ch4_channels:
        plt.plot(scaled_time, df[raw_col_map[ch]], label=ch)

    plt.title("Channels 3 & 4 - Raw Signals")
    plt.xlabel(f"Time ({plot_time_unit})")
    plt.ylabel(f"Amplitude ({yunit})")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print("CH3 and CH4 are not present in this dataset.")

## 5. Plot Raw vs. Filtered Signal Comparison

Plots comparison of Raw vs. Filtered data side-by-side for each channel that has pre-filtered data available in the CSV.

In [ ]:
common_channels = [ch for ch in raw_col_map.keys() if ch in filtered_col_map]

if common_channels:
    for ch in common_channels:
        plt.figure(figsize=(12, 5))
        plt.plot(scaled_time, df[raw_col_map[ch]], ':', label=f"{ch} (Raw)", alpha=0.3)
        plt.plot(scaled_time, df[filtered_col_map[ch]], '-', label=f"{ch} (Filtered)")
        
        plt.title(f"Raw vs. Filtered Signal (from CSV) - {ch}")
        plt.xlabel(f"Time ({plot_time_unit})")
        plt.ylabel(f"Amplitude ({yunit})")
        plt.grid(True)
        plt.legend()
        plt.show()
else:
    print("No pre-filtered channels found in this CSV dataset.")

## 6. Interactive Low-Pass Filtering

Apply a Butterworth low-pass filter dynamically to the raw data columns. You can adjust the cutoff frequency (`cutoff_freq`) below to clean high-frequency noise from any loaded dataset.

In [ ]:
import scipy.signal

# 1. Calculate Sampling Parameters dynamically from the time data
time_diffs = np.diff(time_axis)
mean_dt = np.mean(time_diffs)
fs = 1.0 / mean_dt
nyquist = fs / 2.0

# Configure filter parameters (adjust cutoff_freq to smooth more/less)
cutoff_freq = 0.05 * nyquist  # 5% of Nyquist frequency
filter_order = 4

print(f"Sampling Frequency (fs): {fs:.2f} Hz")
print(f"Nyquist Frequency: {nyquist:.2f} Hz")
print(f"Applying Low-pass Cutoff Frequency (fc): {cutoff_freq:.2f} Hz")

# Design Butterworth low-pass filter
b, a = scipy.signal.butter(filter_order, cutoff_freq, fs=fs, btype='low')

# 2. Filter raw channels dynamically
interactive_filtered = {}
for ch, col in raw_col_map.items():
    interactive_filtered[ch] = scipy.signal.filtfilt(b, a, df[col].to_numpy())

# 3. Plot Comparison of Raw vs. Interactively Filtered Data
for ch in interactive_filtered.keys():
    plt.figure(figsize=(12, 5))
    plt.plot(scaled_time, df[raw_col_map[ch]], ':', label=f"{ch} (Raw)", alpha=0.3)
    plt.plot(scaled_time, interactive_filtered[ch], '-', label=f"{ch} (Filtered)")
    
    plt.title(f"Interactive Low-Pass Filter Comparison - {ch}")
    plt.xlabel(f"Time ({plot_time_unit})")
    plt.ylabel(f"Amplitude ({yunit})")
    plt.grid(True)
    plt.legend()
    plt.show()
